In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score
)

# =========================================================
# Config
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED   = 42
EPOCHS = 20
BATCH  = 128
LR     = 1e-3

MODELS   = ["kNN", "LOF", "IF", "HBOS", "AE"]
DATASETS = ["NSL_KDD", "UNSW_NB15"]
PREP     = "Quantile"

torch.manual_seed(SEED)
np.random.seed(SEED)

# =========================================================
# 데이터 로드
# =========================================================
dfs = {}
for model in MODELS:
    for dataset in DATASETS:
        for split in ["train80", "abnormal", "test"]:
            path = f'{model}_meta_{PREP}/{dataset}_{model}_{split}.csv'
            dfs[f'{split}_{dataset}_{model}'] = pd.read_csv(path)

# =========================================================
# 모델 정의
# =========================================================
class OneLayerNN(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

# =========================================================
# 파이프라인
# =========================================================
SCORE_COLS = [f"score_{m}" for m in MODELS]
results    = []
print(f"\nUsing device: {DEVICE}\n")

for dataset in DATASETS:
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # DataFrame 조합
    parts_train, parts_test = [], []
    for i, model in enumerate(MODELS):
        train_part = (
            pd.concat([dfs[f'train80_{dataset}_{model}'], dfs[f'abnormal_{dataset}_{model}']], ignore_index=True)
            .rename(columns={"anomaly_score": f"score_{model}"})
        )
        test_part = dfs[f'test_{dataset}_{model}'].rename(columns={"anomaly_score": f"score_{model}"})

        if i == 0:
            parts_train.append(train_part)
            parts_test.append(test_part)
        else:
            parts_train.append(train_part[[f"score_{model}"]])
            parts_test.append(test_part[[f"score_{model}"]])

    df_train = pd.concat(parts_train, axis=1).reset_index(drop=True)
    df_test  = pd.concat(parts_test,  axis=1).reset_index(drop=True)

    # 전처리
    qt      = QuantileTransformer(output_distribution="normal", random_state=SEED)
    X_train = qt.fit_transform(df_train[SCORE_COLS].values).astype(np.float32)
    X_test  = qt.transform(df_test[SCORE_COLS].values).astype(np.float32)
    y_train = df_train["label"].values.astype(np.float32)
    y_test  = df_test["label"].values.astype(np.float32)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=BATCH, shuffle=True)
    test_loader  = DataLoader(TensorDataset(torch.tensor(X_test),  torch.tensor(y_test)),  batch_size=BATCH, shuffle=False)

    # 학습
    nn_model  = OneLayerNN(input_dim=len(MODELS), hidden_dim=10).to(DEVICE)
    optimizer = torch.optim.Adam(nn_model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.BCELoss()

    for _ in range(EPOCHS):
        nn_model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            criterion(nn_model(X_batch), y_batch).backward()
            optimizer.step()

    # 평가
    nn_model.eval()
    probs, y_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            probs.extend(nn_model(X_batch.to(DEVICE)).cpu().numpy())
            y_true.extend(y_batch.numpy())
    probs, y_true = np.array(probs), np.array(y_true)
    preds_bin     = (probs >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, preds_bin, labels=[0, 1]).ravel()
    results.append({
        "Dataset":     dataset,
        "Recall":      recall_score(y_true, preds_bin, zero_division=0),
        "Precision":   precision_score(y_true, preds_bin, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "F1-score":    f1_score(y_true, preds_bin, zero_division=0),
        "Accuracy":    accuracy_score(y_true, preds_bin),
        "AUC":         roc_auc_score(y_true, probs),
    })

# =========================================================
# 결과 출력
# =========================================================
COLS    = ["Precision", "Recall", "Specificity", "F1-score", "Accuracy", "AUC"]
COLS_KR = ["정밀도",    "민감도", "특이도",       "F1 점수",  "정확도",   "AUC"]
W       = 72

print("=" * W)
print(f"  메타러닝 결과  (전처리: {PREP} 고정)")
print("=" * W)
print(f"  {'':16s}" + "".join(f"{k:>8}" for k in COLS_KR))
print("-" * W)
for r in results:
    print(f"  {r['Dataset']:<16}" + "".join(f"{r[c]:>8.2f}" for c in COLS))
print("-" * W)


Using device: cpu

  메타러닝 결과  (전처리: Quantile 고정)
                       정밀도     민감도     특이도   F1 점수     정확도     AUC
------------------------------------------------------------------------
  NSL_KDD             0.96    0.89    0.95    0.93    0.92    0.98
  UNSW_NB15           0.77    0.97    0.65    0.86    0.83    0.95
------------------------------------------------------------------------
